# Verificación de `nflreadpy` como fuente de datos — WR semanal

## Qué responde este notebook

Este proyecto predice, para cada receptor (WR) de la NFL, cuánto va a producir la próxima semana
(recepciones, yardas, touchdowns). Para eso necesita datos: quién es cada jugador, qué partidos hay,
qué hizo cada quien cada semana, quién es titular y el detalle jugada por jugada.

Todos vienen de **`nflverse`**, un proyecto de código abierto mantenido por la comunidad (no una
empresa con contrato ni soporte). Se leen desde Python con dos librerías: `nfl_data_py`, con la que se
hicieron los notebooks originales del proyecto (guardados en `weekly/wr/notebooks/_reference/`) y que
hoy está descontinuada, y `nflreadpy`, su sucesora activa y la que se usa de aquí en adelante.

**Antes de construir el pipeline sobre `nflreadpy` hay que comprobar que sea confiable**: no basta con
que funcione, hay que revisar que sus datos estén completos, correctos y disponibles a tiempo. Este
notebook lo hace sobre las 5 fuentes que podría usar el pipeline. Si alguna tiene datos incompletos,
duplicados o con retraso, el modelo entero hereda ese problema sin que nadie lo note.

| Fuente (función de `nflreadpy`) | Qué responde |
|---|---|
| `load_players()` | ¿Quién es cada jugador? (identidad) |
| `load_schedules()` | ¿Qué partidos hay, cuándo, con qué contexto (apuestas, clima)? |
| `load_player_stats()` | ¿Qué produjo cada jugador, cada semana? (las cifras que se quieren predecir) |
| `load_pbp()` | ¿Qué pasó en cada jugada del partido? |
| `load_depth_charts()` | ¿Quién es titular esta semana? |

Los términos de fútbol americano y las columnas que aparecen en el notebook se definen en el
[glosario de variables](../../../docs/data/GLOSARIO_VARIABLES.md).

## Cómo usar este notebook

Se ejecuta dentro del contenedor Docker del proyecto (`docker compose up python`), que ya tiene
`nflreadpy` instalado. `SEASONS` (antes de la Parte 2) es el único parámetro que normalmente se
cambia: hoy son 2024 y 2025, dos temporadas completas y ya cerradas, pero funciona con cualquier otro
rango.

Los resultados están incorporados a la documentación del proyecto: [`docs/data/DATASHEET.md`](../../../docs/data/DATASHEET.md)
(sección "Auditoría de `nflreadpy` por fuente") y [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md). Este notebook es la
evidencia ejecutable detrás de esos documentos: si algo cambia (nueva versión de `nflreadpy`, más
años), se vuelve a correr y se compara.

## Parte 1 — ¿La temporada en curso llega a tiempo?

Cada semana, `nflverse` tiene que armar y publicar la tabla resumen de estadísticas de esa semana, y
ese proceso puede atrasarse unos días respecto al partido real. Si el pipeline depende de esa tabla y
se atrasa, no puede generar a tiempo la predicción de la semana en curso.

La librería anterior, `nfl_data_py`, tiene justo ese problema: su función `import_weekly_data()`
regresa error para la temporada en curso (también está registrado en la fila de vigencia del
[Datasheet](../../../docs/data/DATASHEET.md)). La pregunta de esta sección: **¿la función equivalente de `nflreadpy`,
`load_player_stats(summary_level="week")`, tiene el mismo problema?**

Se prueba llamando a ambas el mismo día, para la misma temporada, y comparando qué regresa cada una.

In [1]:
import datetime
import nfl_data_py as nfldp
import nflreadpy as nfl

print("Hoy:", datetime.date.today())

print()
print("=== nfl_data_py.import_weekly_data([2026]) — la libreria vieja ===")
try:
    wk_old = nfldp.import_weekly_data([2026])
    print("filas:", len(wk_old), "| semanas:", sorted(wk_old["week"].unique().tolist()))
except Exception as e:
    print("ERROR:", type(e).__name__, "-", str(e)[:200])

print()
print("=== nflreadpy.load_player_stats([2026], summary_level='week') — la libreria nueva ===")
ws_2026 = nfl.load_player_stats([2026], summary_level="week")
print("filas:", ws_2026.height, "| semanas:", sorted(ws_2026["week"].unique().to_list()))

Hoy: 2026-10-05

=== nfl_data_py.import_weekly_data([2026]) — la libreria vieja ===
ERROR: HTTPError - HTTP Error 404: Not Found

=== nflreadpy.load_player_stats([2026], summary_level='week') — la libreria nueva ===
filas: 4302 | semanas: [1, 2, 3, 4]


**Resultado:** `import_weekly_data` falla (error 404): la tabla resumen de la temporada en curso aún no
existe del lado viejo. `load_player_stats`, en cambio, sí trae datos, hasta la semana más reciente ya
jugada. La librería nueva no hereda el retraso de la vieja.

Esto simplifica el pipeline: no hace falta calcular las estadísticas a mano desde el detalle de
jugadas (`load_pbp`) para evitar el retraso, porque `load_player_stats` sirve tanto para la historia
como para la semana en curso.

In [3]:
nfl.load_player_stats?

Signature:
nfl.load_player_stats(
    seasons: int | list[int] | bool | None = None,
    summary_level: Literal['week', 'reg', 'post', 'reg+post'] = 'week',
) -> polars.dataframe.frame.DataFrame
Docstring:
Load NFL player statistics.

Args:
    seasons: Season(s) to load. If None, loads current season.
            If True, loads all available data.
            If int or list of ints, loads specified season(s).
    summary_level: Summary level ("week", "reg", "post", "reg+post").

Returns:
    Polars DataFrame with player statistics.

See Also:
    <https://nflreadr.nflverse.com/reference/load_player_stats.html>

Data Dictionary:
    <https://nflreadr.nflverse.com/articles/dictionary_player_stats.html>
File:      /usr/local/lib/python3.13/site-packages/nflreadpy/load_stats.py
Type:      function

In [2]:
ws_2026

player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,…,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
str,str,str,str,str,str,i32,i32,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,str,str,str,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64
"""00-0023459""","""A.Rodgers""","""Aaron Rodgers""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2026,1,"""REG""","""2026_01_ATL_PIT""","""PIT""","""ATL""",24,40,221,1,0,2,-15,0,0,407,121,10,-7.838145,1.234483,0,0.542998,9,5,3,0,3,-3,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,12.54,12.54
"""00-0025565""","""N.Folk""","""Nick Folk""","""K""","""SPEC""","""https://static.www.nfl.com/ima…",2026,1,"""REG""","""2026_01_ATL_PIT""","""ATL""","""PIT""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,1,1,0,"""51;43""","""54;45""",null,94,99,0,1,1,0,0,1.0,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0026190""","""C.Campbell""","""Calais Campbell""","""DE""","""DL""","""https://static.www.nfl.com/ima…",2026,1,"""REG""","""2026_01_BAL_IND""","""BAL""","""IND""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0026498""","""M.Stafford""","""Matthew Stafford""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2026,1,"""REG""","""2026_01_SF_LA""","""LA""","""SF""",15,25,155,0,1,0,0,0,0,232,85,5,-5.611142,-4.68852,0,0.668103,6,3,2,1,2,-1,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,4.1,4.1
"""00-0027969""","""C.Heyward""","""Cameron Heyward""","""DE""","""DL""","""https://static.www.nfl.com/ima…",2026,1,"""REG""","""2026_01_ATL_PIT""","""PIT""","""ATL""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""00-0041556""","""R.Height""","""Romello Height""","""DE""","""DL""","""https://static.www.nfl.com/ima…",2026,4,"""REG""","""2026_04_DEN_SF""","""SF""","""DEN""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0041557""","""G.Halton""","""Gracen Halton""","""DT""","""DL""","""https://static.www.nfl.com/ima…",2026,4,"""REG""","""2026_04_DEN_SF""","""SF""","""DEN""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0041558""","""S.Roush""","""Sam Roush""","""TE""","""TE""","""https://static.www.nfl.com/ima…",2026,4,"""REG""","""2026_04_NYJ_CHI""","""CHI""","""NYJ""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null

## Parte 2 — Calidad de cada fuente, con volumen real

La Parte 1 solo probó las pocas semanas ya jugadas de la temporada en curso. Eso basta para saber si
los datos llegan a tiempo, pero no si están *bien*: huecos raros, duplicados o identidades mezcladas
solo se notan con volumen. Por eso aquí se descargan **dos temporadas completas y ya cerradas** (2024 y
2025; cerradas para no mezclar este análisis con el retraso de la Parte 1) y se revisa cada fuente con
6 dimensiones de calidad de datos: exactitud, completitud, consistencia, validez, vigencia y unicidad
(se definen en el [Datasheet](../../../docs/data/DATASHEET.md)).

In [8]:
import polars as pl

SEASONS = [2024, 2025]  # cambiar aqui para correr esto sobre otro rango de anios

### Fuente 1 — `load_players()`: identidad de jugador

Es la tabla maestra de todos los jugadores que ha habido en la NFL, cada uno con un identificador
único, `gsis_id`, que sirve para cruzar las demás fuentes sin ambigüedad. Los archivos de ADP
(*average draft position*, la posición promedio en que se elige a cada jugador en los drafts de
fantasy) de la carpeta `data/` también traen `gsis_id`, y ahí un mismo identificador aparece repetido
en 2 o 3 jugadores distintos, lo que mezclaría sus estadísticas (detalle en
[`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)). Aquí se revisa si `load_players()` tiene el mismo problema.

In [9]:
players = nfl.load_players()
print("filas (todos los jugadores historicos):", players.height, "| columnas:", players.width)

gsis = players.filter(players["gsis_id"].is_not_null())
dup = gsis["gsis_id"].value_counts().filter(pl.col("count") > 1)
print("gsis_id no nulos:", gsis.height, "/", players.height)
print("gsis_id que se repiten en mas de una fila (0 es lo esperado):", dup.height)

cols_clave = ["gsis_id", "display_name", "position", "birth_date", "draft_year", "draft_round"]
for c in cols_clave:
    pct = round(players[c].null_count() / players.height * 100, 2)
    print(f"  % de filas sin dato en '{c}': {pct}%")

filas (todos los jugadores historicos): 24844 | columnas: 39
gsis_id no nulos: 24844 / 24844
gsis_id que se repiten en mas de una fila (0 es lo esperado): 0
  % de filas sin dato en 'gsis_id': 0.0%
  % de filas sin dato en 'display_name': 0.0%
  % de filas sin dato en 'position': 0.0%
  % de filas sin dato en 'birth_date': 0.13%
  % de filas sin dato en 'draft_year': 49.72%
  % de filas sin dato en 'draft_round': 49.72%


In [11]:
gsis

gsis_id,display_name,common_first_name,first_name,last_name,short_name,football_name,suffix,esb_id,nfl_id,pfr_id,pff_id,otc_id,espn_id,smart_id,birth_date,position_group,position,ngs_position_group,ngs_position,height,weight,headshot,college_name,college_conference,jersey_number,rookie_season,last_season,latest_team,status,ngs_status,ngs_status_short_description,years_of_experience,pff_position,pff_status,draft_year,draft_round,draft_pick,draft_team
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,str,str,str,str,i32,i32,str,str,str,str,i32,str,str,i32,i32,i32,str
"""00-0028830""","""Isaako Aaitui""","""Isaako""","""Isaako""","""Aaitui""",null,null,null,"""AAI622937""",null,"""AaitIs00""","""6998""","""2535""","""14856""","""32004141-4962-2937-61ff-017b18…","""1987-01-25""","""DL""","""NT""",null,null,76,307,"""https://static.www.nfl.com/ima…","""UNLV""",null,"""0""",2011,2014,"""WAS""","""DEV""",null,null,2,"""DI""",null,null,null,null,null
"""00-0038389""","""Israel Abanikanda""","""Israel""","""Israel""","""Abanikanda""","""I.Abanikanda""","""Israel""",null,"""ABA159567""","""56008""","""AbanIs00""","""122999""","""10967""","""4429202""","""32004142-4115-9567-2e24-0eab29…","""2002-10-05""","""RB""","""RB""","""RB""","""RB""",70,225,"""https://static.www.nfl.com/ima…","""Pittsburgh""","""Atlantic Coast Conference""","""25""",2023,2026,"""DAL""","""DEV""","""DEV""","""Practice Squad""",3,"""HB""","""A""",2023,5,143,"""NYJ"""
"""00-0024644""","""Jon Abbate""","""Jon""","""Jon""","""Abbate""",null,null,null,"""ABB051371""",null,null,null,null,null,"""32004142-4205-1371-db95-1abc96…","""1985-06-18""","""LB""","""LB""",null,null,71,245,"""https://static.www.nfl.com/ima…","""Wake Forest""",null,"""67""",2007,2007,"""HOU""","""RES""",null,null,0,null,null,null,null,null,null
"""ABB498348""","""Vince Abbott""","""Vince""","""Vincent""","""Abbott""",null,null,null,"""ABB498348""",null,"""abbotvin01""",null,null,null,"""32004142-4249-8348-e00f-5fbbe6…","""1958-05-31""","""SPEC""","""K""",null,null,71,207,"""https://static.www.nfl.com/ima…","""California State-Fullerton; Wa…",null,"""0""",1987,1988,"""LAC""","""ACT""",null,null,2,null,null,null,null,null,null
"""00-0031021""","""Jared Abbrederis""","""Jared""","""Jared""","""Abbrederis""","""J.Abbrederis""","""Jared""",null,"""ABB650964""","""41405""","""AbbrJa00""","""8811""","""3115""","""16836""","""32004142-4265-0964-fc36-bb0ad7…","""1990-12-17""","""WR""","""WR""","""WR""","""WR""",73,195,"""https://static.www.nfl.com/ima…","""Wisconsin""",null,"""10""",2014,2017,"""DET""","""CUT""","""CUT""",null,4,"""WR""",null,2014,5,176,"""GB"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""DEB622935""","""Case deBruijn""","""Case""","""Case""","""deBruijn""",null,null,null,"""DEB622935""",null,"""deBrCa20""",null,null,null,"""32004445-4262-2935-c44d-416ca8…","""1960-04-11""","""SPEC""","""P""",null,null,72,176,"""https://static.www.nfl.com/ima…","""Idaho State""",null,"""0""",1982,1982,"""KC""","""ACT""",null,null,0,null,null,1982,8,214,"""KC"""
"""VAN516304""","""Mark van Eeghen""","""Mark""","""Mark""","""van Eeghen""",null,null,null,"""VAN516304""",null,"""VanEMa00""",null,null,null,"""32005641-4e51-6304-9ea7-e13b6d…","""1952-04-19""","""RB""","""RB""",null,null,74,223,"""https://static.www.nfl.com/ima…","""Colgate""",null,"""0""",1974,1983,"""NE""","""ACT""",null,null,10,null,null,1974,3,75,"""LV"""
"""00-0041110""","""Jordan van den Berg""","""Jordan""","""Jordan""","""van den Berg""","""J.van den Berg""","""Jordan""",null,"""VAN366267""","""62835""","""vandJo00""","""145085""","""13357""","""4875381""","""32005641-4e36-6267-1a50-c0be45…","""2002-04-12""","""DL""","""DT""","""DL""","""INTERIOR_LINE""",75,310,"""https://static.www.nfl.com/ima…","""Georgia Tech; Penn State; Iowa…","""Atlantic Coast Conference""","""96""",2026,2026,"""CHI""","""ACT""","""ACT""","""Active""",0,"""DI""","""A""",2026,6,213,"""CHI"""


**Lectura:** cero colisiones de `gsis_id`: a diferencia del archivo de ADP, cada identificador
corresponde a un único jugador. Por eso `load_players()` es la fuente de identidad del proyecto.

`draft_year` y `draft_round` faltan en ~50% de los jugadores, y ese hueco sí tiene sentido: quien entró
a la liga sin ser drafteado no tiene año ni ronda de draft. Es un "nulo estructural" (el dato no
aplica), no un error, y no requiere limpieza.

### Fuente 2 — `load_schedules()`: el calendario de partidos

Trae, para cada partido, cuándo se jugó y quién contra quién, además del contexto del juego: la línea
de apuestas (`spread_line`: ventaja en puntos que el mercado le da al local; `total_line`: puntos
totales que espera entre ambos equipos) y el clima (`temp`, `wind`). Ese contexto podría servir como
variable del modelo; aquí solo se revisa que los datos estén completos y bien formados, y si aporta
algo para predecir se evalúa en la sección 4 de `2.3_eda_general.ipynb`. Sus definiciones están en el
[glosario](../../../docs/data/GLOSARIO_VARIABLES.md) (sección 9 y apéndice B.2).

In [12]:
sched = nfl.load_schedules(SEASONS)
print("filas:", sched.height, "| si solo fuera temporada regular esperariamos:", 272 * len(SEASONS))
print("semanas presentes:", sorted(sched["week"].unique().to_list()))
print("tipos de partido presentes (game_type):", sorted(sched["game_type"].unique().to_list()))

for c in ["spread_line", "total_line", "temp", "wind"]:
    pct = round(sched[c].null_count() / sched.height * 100, 2)
    print(f"  % de partidos sin dato en '{c}': {pct}%")

filas: 570 | si solo fuera temporada regular esperariamos: 544
semanas presentes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
tipos de partido presentes (game_type): ['CON', 'DIV', 'REG', 'SB', 'WC']
  % de partidos sin dato en 'spread_line': 0.0%
  % de partidos sin dato en 'total_line': 0.0%
  % de partidos sin dato en 'temp': 34.74%
  % de partidos sin dato en 'wind': 34.74%


**Lectura:** salen más partidos de los esperables (272 por temporada regular: 32 equipos × 17
partidos ÷ 2) porque `load_schedules()` trae *también* los playoffs por default (semanas 19 a 22:
Wild Card, Divisional, Conferencia, Super Bowl). No es un error de la fuente: al construir el pipeline
hay que pedir explícitamente solo temporada regular (`game_type == "REG"`); de lo contrario, un
promedio por semana mezclaría partidos de playoffs sin querer.

`spread_line` y `total_line` están completos. `temp` y `wind` faltan en ~35% de los partidos, y de
nuevo tiene sentido: los partidos bajo techo (domo) no tienen clima que reportar. Es un "no aplica",
no un hueco de datos.

### Fuente 3 — `load_player_stats(summary_level="week")`: lo que produjo cada jugador, cada semana

Es la fuente más importante del pipeline: de aquí sale el número que se quiere predecir (recepciones,
yardas y touchdowns de cada WR, cada semana). Si tiene duplicados o huecos raros, el modelo aprendería
de datos incorrectos sin que se note.

In [5]:
ws = nfl.load_player_stats(SEASONS, summary_level="week")
wr = ws.filter(ws["position"] == "WR")
print("filas totales (todas las posiciones):", ws.height, "| filas solo WR:", wr.height)

# Cada jugador deberia aparecer una sola vez por temporada y semana.
# Si aparece 2+ veces, algo esta mal (o se sumarian mal sus stats, o se duplicarian filas).
dup = ws.group_by(["player_id", "season", "week"]).len().filter(pl.col("len") > 1)
print("combinaciones (jugador, temporada, semana) que aparecen mas de una vez (0 es lo esperado):", dup.height)

cols_wr = ["receptions", "receiving_yards", "receiving_tds", "targets",
           "target_share", "racr", "air_yards_share", "wopr", "receiving_epa"]
for c in cols_wr:
    pct = round(wr[c].null_count() / wr.height * 100, 2)
    print(f"  % de filas WR sin dato en '{c}': {pct}%")

filas totales (todas las posiciones): 38405 | filas solo WR: 5195
combinaciones (jugador, temporada, semana) que aparecen mas de una vez (0 es lo esperado): 0
  % de filas WR sin dato en 'receptions': 0.0%
  % de filas WR sin dato en 'receiving_yards': 0.0%
  % de filas WR sin dato en 'receiving_tds': 0.0%
  % de filas WR sin dato en 'targets': 0.0%
  % de filas WR sin dato en 'target_share': 0.0%
  % de filas WR sin dato en 'racr': 15.42%
  % de filas WR sin dato en 'air_yards_share': 0.0%
  % de filas WR sin dato en 'wopr': 0.0%
  % de filas WR sin dato en 'receiving_epa': 15.23%


**Lectura:** cero filas duplicadas: cada WR aparece una vez por semana, que es lo que se necesita para
que cada fila de entrenamiento sea un único partido de un jugador. Las columnas principales
(recepciones, yardas, `target_share`) están completas al 100%.

`racr` y `receiving_epa` sí tienen huecos (~15%). Antes de decidir qué hacer hay que entender por qué,
y se investiga en la siguiente celda. Qué mide cada columna está en el
[glosario de variables](../../../docs/data/GLOSARIO_VARIABLES.md) (sección 6); en corto, `target_share` es la fracción de los pases de su
equipo dirigidos a él, `racr` son las yardas ganadas entre las yardas aéreas y `receiving_epa` los
puntos esperados que aportan las jugadas en que fue objetivo.

In [6]:
null_racr = wr.filter(pl.col("racr").is_null())
print("filas WR con 'racr' vacio:", null_racr.height)
print("  de esas, jugadores que esa semana tuvieron 0 targets (targets == 0):",
      null_racr.filter(pl.col("targets") == 0).height)
print("  de esas, con targets > 0 (esto SI seria raro de explicar):",
      null_racr.filter(pl.col("targets") > 0).height)

filas WR con 'racr' vacio: 801
  de esas, jugadores que esa semana tuvieron 0 targets (targets == 0): 796
  de esas, con targets > 0 (esto SI seria raro de explicar): 5


**Conclusión:** 796 de 801 filas (99.4%) con `racr` vacío son de jugadores que esa semana no tuvieron
ni un solo pase dirigido a ellos (un *target*): sin jugadas de las que partir, el cálculo no aplica,
igual que el clima bajo techo. Solo 5 filas (0.1% de las filas WR) quedan sin explicación clara; es un
residuo tan pequeño que se documenta y no bloquea nada.

**Otro hallazgo, mismo patrón que el calendario:** `load_player_stats()` también mezcla playoffs por
default (`season_type` puede ser `REG` o `POST`). Se revisa antes de dar por buena la fuente.

In [7]:
print("season_type presentes en las stats semanales:", ws["season_type"].unique().to_list())
print("semanas presentes (deberian llegar hasta 22 si incluye playoffs):",
      sorted(ws["week"].unique().to_list()))

season_type presentes en las stats semanales: ['REG', 'POST']
semanas presentes (deberian llegar hasta 22 si incluye playoffs): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]


### Fuente 4 — `load_pbp()`: el detalle de cada jugada

Trae una fila por cada jugada de cada partido (no por jugador-semana). En los notebooks originales se
usaba para calcular a mano `target_share`, pero `load_player_stats` ya trae esa columna (se revisó en
la Fuente 3). Se audita igualmente, por si hace falta calcular algo que la tabla semanal no trae.

In [8]:
pbp = nfl.load_pbp(SEASONS)
print("filas (una por jugada):", pbp.height)
for c in ["receiver_id", "receiver_player_name", "pass_attempt", "air_yards"]:
    pct = round(pbp[c].null_count() / pbp.height * 100, 2)
    print(f"  % de jugadas sin dato en '{c}': {pct}%")
print("season_type presentes:", pbp["season_type"].unique().to_list())

filas (una por jugada): 98263
  % de jugadas sin dato en 'receiver_id': 61.88%
  % de jugadas sin dato en 'receiver_player_name': 63.94%
  % de jugadas sin dato en 'pass_attempt': 3.08%
  % de jugadas sin dato en 'air_yards': 62.4%
season_type presentes: ['POST', 'REG']


**Lectura:** ~62% de las jugadas no tienen receptor, y tiene sentido: un acarreo, una patada o un
despeje no involucran a ningún receptor. Es otro "no aplica" estructural, no un problema.

**Mismo patrón de playoffs:** `load_pbp()` también trae `POST` junto con `REG`, y se filtra igual que
en las otras dos fuentes.

### Fuente 5 — `load_depth_charts()`: quién es titular esta semana

Una alineación (*depth chart*) es la lista que publica cada equipo con sus titulares y suplentes por
posición (ver [glosario](../../../docs/data/GLOSARIO_VARIABLES.md), sección 1). Aquí apareció algo que **no** se explica con "no aplica":
el hallazgo más serio de esta auditoría.

In [9]:
depth = nfl.load_depth_charts(SEASONS)
print("filas:", depth.height)
for c in ["position", "depth_team", "pos_abb"]:
    pct = round(depth[c].null_count() / depth.height * 100, 2)
    print(f"  % de filas sin dato en '{c}': {pct}%")

filas: 591527
  % de filas sin dato en 'position': 93.69%
  % de filas sin dato en 'depth_team': 93.69%
  % de filas sin dato en 'pos_abb': 6.31%


A primera vista, un ~94% de huecos en `position`/`depth_team` se ve como un problema grave de
calidad. Pero antes de concluir eso, hay que preguntarse lo mismo que en las otras fuentes:
**¿ese hueco tiene una explicación estructural?** Para saberlo, se pide cada temporada por
separado (en vez de las dos juntas) y se comparan las columnas que trae cada una:

In [10]:
for yr in [2022, 2023, 2024, 2025, 2026]:
    d = nfl.load_depth_charts([yr])
    print(f"{yr}: filas={d.height} | columnas={d.columns}")

2022: filas=37780 | columnas=['season', 'club_code', 'week', 'game_type', 'depth_team', 'last_name', 'first_name', 'football_name', 'formation', 'gsis_id', 'jersey_number', 'position', 'elias_id', 'depth_position', 'full_name']
2023: filas=37327 | columnas=['season', 'club_code', 'week', 'game_type', 'depth_team', 'last_name', 'first_name', 'football_name', 'formation', 'gsis_id', 'jersey_number', 'position', 'elias_id', 'depth_position', 'full_name']
2024: filas=37312 | columnas=['season', 'club_code', 'week', 'game_type', 'depth_team', 'last_name', 'first_name', 'football_name', 'formation', 'gsis_id', 'jersey_number', 'position', 'elias_id', 'depth_position', 'full_name']


2025: filas=554215 | columnas=['dt', 'team', 'player_name', 'espn_id', 'gsis_id', 'pos_grp_id', 'pos_grp', 'pos_id', 'pos_name', 'pos_abb', 'pos_slot', 'pos_rank']
2026: filas=581172 | columnas=['dt', 'team', 'player_name', 'espn_id', 'gsis_id', 'pos_grp_id', 'pos_grp', 'pos_id', 'pos_name', 'pos_abb', 'pos_slot', 'pos_rank']


**La causa real:** no es un hueco de datos, sino un **cambio de esquema** entre temporadas. De 2022 a
2024 las alineaciones traen las columnas `season`/`week`/`game_type`/`position`/`depth_team`, ya
organizadas por semana. **De 2025 en adelante (y se confirmó que también para 2026, la temporada en
curso cuando se ejecutó este notebook) `nflreadpy` cambia a otro formato**:
`dt`/`team`/`pos_grp`/`pos_rank`/`pos_abb`, que ni siquiera tiene columnas de temporada o semana.

Al pedir los dos años juntos (`load_depth_charts([2024, 2025])`) la librería junta ambos formatos en
una sola tabla: las filas de 2025 no tienen las columnas del formato viejo (por eso salen "vacías") y
viceversa. El dato no falta; vive en una columna con otro nombre.

**Por qué importa para el pipeline:** los notebooks originales
(`weekly/wr/notebooks/_reference/wrs_rec_tds.ipynb` y los demás de esa carpeta) usan
`position`/`depth_team` del formato viejo para identificar a los 3 titulares de cada equipo. Si el
pipeline pide la temporada en curso esperando esas columnas, falla o, peor, produce una tabla llena de
vacíos sin ningún error visible. Por eso debe reconocer qué formato le tocó según el año y traducir
uno al otro antes de usarlo (detalle en [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)).

## Cierre

Qué se puede usar tal cual y qué necesita trabajo extra:

| Fuente | ¿Confiable tal cual? | Qué hacer en el pipeline |
|---|---|---|
| `load_players()` | Sí — identidad limpia, sin colisiones | Usar para identidad, en vez del archivo de ADP |
| `load_schedules()` | Sí, con un ajuste | Filtrar `game_type == "REG"` para no mezclar playoffs |
| `load_player_stats(summary_level="week")` | Sí, con un ajuste | Filtrar `season_type == "REG"` (mismo problema que el calendario); incluye la semana en curso sin retraso |
| `load_pbp()` | Sí, con un ajuste | Filtrar `season_type == "REG"`; los huecos que tiene son esperados |
| `load_depth_charts()` | No tal cual | Construir un adaptador que traduzca el formato viejo (≤2024) y el nuevo (2025+) a las mismas columnas antes de usarlos juntos |

Los filtros y el adaptador de esta tabla están implementados en `weekly/wr/src/data.py`; el notebook
siguiente explica cómo se construyó cada uno.

Siguiente: [`1.2_construccion_datos.ipynb`](1.2_construccion_datos.ipynb)